# 第 2 章 · ADK 工程骨架与 CLI

> **你已会**：第 1 章的 Agent、Runner、Session、Event。
>
> **读完后你能**：① 建出一个 `adk` 能识别的标准工程；② 用 `adk run` / `adk web` / `adk api_server` 三种方式启动它；③ 用 HTTP 调用它。
>
> **本章不碰**：云端部署、评估集（第 10 章）。

**怎么读**

- 主线（约 20 分钟）：§1 约定 → §2 目录 → §5 命令。
- 本章只有一个代码格，其余是你会在终端执行的命令。

---

## 1. 为什么需要标准工程

第 1 章把所有代码写在 notebook 里。能跑，但有三件事做不了：

- `adk web`、`adk run`、`adk api_server` 都按固定约定寻找 Agent；
- `adk eval` 需要一个可导入的模块入口；
- 代码无法被版本化、复用和部署。

ADK 的约定只有一句话：**一个目录 = 一个 Agent 应用；目录里的 `agent.py` 暴露 `root_agent`。**

---

## 2. 目录结构

本仓库已经准备了一个最小示例：

```text
google-adk/agent_demo/
├── __init__.py      # from . import agent
├── agent.py         # 定义 root_agent
└── （可选）tools.py  # 工具函数可以拆出去
```

| 文件 | 作用 | 必须吗 |
|---|---|---|
| `agent.py` | 定义名为 `root_agent` 的 Agent | 是 |
| `__init__.py` | 让目录成为可导入的包，并导入 `agent` | 是 |
| `.env` | 存放 `DEEPSEEK_API_KEY`，不要提交 | 推荐 |

---

## 3. 三个文件

`agent.py` 的核心就是一个名为 `root_agent` 的变量：

```python
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

root_agent = Agent(
    name="study_buddy",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是一名学习计划助手……",
    description="帮助学习者制定可执行的学习计划",
)
```

`__init__.py` 只有一行：

```python
from . import agent
```

密钥只从环境变量读取。不要在 `agent.py` 里写 `api_key=...`，也不要把 `.env` 提交到仓库。

---

## 4. 先验证能加载

下面不启动服务，只把 `root_agent` 导进来检查配置：

In [1]:
import sys
from pathlib import Path

# 兼容"从仓库根目录启动"和"从 google-adk 目录启动"两种方式
root = Path.cwd()
if not (root / "agent_demo").exists():
    root = root / "google-adk"

sys.path.insert(0, str(root))
from agent_demo.agent import root_agent

print("name       :", root_agent.name)
print("description:", root_agent.description)
print("model      :", root_agent.model.model)

name       : study_buddy
description: 帮助学习者制定可执行的学习计划
model      : deepseek/deepseek-chat


---

## 5. 三个 CLI 命令

| 命令 | 用途 | 什么时候用 |
|---|---|---|
| `adk run <agent_dir>` | 终端里跑一次或进入交互 | 快速验证 |
| `adk web <agents_dir>` | 带 Web UI 的调试服务 | 看事件流、State、Graph |
| `adk api_server <agents_dir>` | 纯 HTTP API | 给前端或其他服务调用 |

注意参数的区别：`adk run` 指向**一个 Agent 目录**；`adk web` 和 `adk api_server` 指向**包含多个 Agent 子目录的父目录**（也可以直接指向单个 Agent 目录）。

在仓库根目录执行：

```bash
uv run adk run google-adk/agent_demo "用一句话介绍你自己"
```

```bash
uv run adk web google-adk --port 8080
```

```bash
uv run adk api_server google-adk --host 127.0.0.1 --port 8000
```

> 💡 `adk web` 启动后打开浏览器，重点看四个面板：**Events**（完整事件 JSON）、**State**（状态变化）、**Graph**（多 Agent 结构）、**Trace**（耗时）。

---

## 6. 用 HTTP 调用

`adk api_server` 启动后，先建会话：

```bash
curl -X POST http://127.0.0.1:8000/apps/agent_demo/users/student/sessions \
  -H "Content-Type: application/json" \
  -d '{"session_id": "s1"}'
```

再发消息：

```bash
curl -X POST http://127.0.0.1:8000/run \
  -H "Content-Type: application/json" \
  -d '{"app_name":"agent_demo","user_id":"student","session_id":"s1","new_message":{"role":"user","parts":[{"text":"用一句话介绍你自己"}]}}'
```

返回值是一个**事件列表**，不是一段字符串——这正是第 1 章说的"ADK 的世界观"。

> 📌 `/run` 返回完整事件列表；需要流式输出时用 `/run_sse`。会话不存在会返回 404，先建会话再调用。

---

## 7. 四个常见错误

| 现象 | 原因 | 处理 |
|---|---|---|
| `root_agent not found` | `agent.py` 里没有名为 `root_agent` 的变量 | 改名或导出 |
| `ModuleNotFoundError: agent` | 缺 `__init__.py`，或目录层级指错 | 补文件，确认传给 CLI 的是父目录 |
| 模型 401 / 认证失败 | `DEEPSEEK_API_KEY` 没注入进程 | 用 `--env-file` 或先导出环境变量 |
| 端口被占用 | 8000/8080 已被使用 | 换 `--port` |

---

## 📌 用五句话复述本章

1. **一个目录就是一个 Agent 应用**，`agent.py` 里必须有 `root_agent`。
2. **`__init__.py` 不是可有可无**，它让目录成为可导入的包。
3. **`adk run` 指向单个 Agent，`adk web` / `adk api_server` 通常指向父目录。**
4. **`adk api_server` 暴露的是事件列表**，调用前要先建会话。
5. **密钥只从环境变量读**，永远不写进代码或提交到仓库。

---

## 🧪 本章练习

### 1. 把第 1 章的助手搬进标准工程（基础）

- [ ] 新建一个目录，包含 `agent.py` 和 `__init__.py`
- [ ] `adk run` 能回答一个问题
- [ ] `adk web` 能看到完整事件流

### 2. 加一个工具（进阶）

- [ ] 在工程里加入第 4 章的天气工具
- [ ] 用 `adk web` 观察 `function_call` 与 `function_response`
- [ ] 确认工具文件拆分后仍能被导入

### 3. 换成 SQLite 会话（进阶）

- [ ] 用 `--session_service_uri` 指向 SQLite
- [ ] 重启服务后，同一个 `session_id` 仍能继续对话
- [ ] 说明这与第 6 章 `DatabaseSessionService` 的关系

### 4. 写一个启动脚本（工程）

- [ ] 一条命令完成：检查密钥、启动 `adk api_server`
- [ ] 缺密钥时给出清晰错误，而不是 401
- [ ] 支持自定义端口和 Agent 目录